# 3. Na atenção primária, o que acontece depois do atendimento (Tabela 1)

Reproduz a tabela de conduta do site. O relatório do SISAB, com a coluna "Conduta" no lugar do diagnóstico, foi
consultado separadamente para cada diagnóstico. Cada conduta, somada de jan/2025 até o último mês, é dividida pelos
atendimentos do mesmo diagnóstico no mesmo período. Um atendimento pode ter mais de uma conduta, então as linhas não
somam 100%. O registro não diz qual é o serviço especializado.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ (o notebook roda de dentro de notebooks/)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.config import *
from src.util import *

In [ ]:
INICIO_CONDUTA = "202501"
conduta = pl.read_parquet(RAW / "atencao_primaria" / "sisab_conduta.parquet")
conduta = conduta.filter(pl.col("cmp").is_between(pl.lit(INICIO_CONDUTA), pl.lit(FIM_APS)))
aps = pl.read_parquet(RAW / "atencao_primaria" / "sisab_aps.parquet")
aps = aps.filter((pl.col("recorte") == "total") & pl.col("cmp").is_between(pl.lit(INICIO_CONDUTA), pl.lit(FIM_APS)))

# nomes das condutas como vêm (truncados) no relatório -> coluna da tabela
CONDUTAS = {"Alta do episódio": "Alta", "Retorno para consulta agendada": "Retorno agendado",
            "Retorno p/ cuidado continuado/": "Retorno para cuidado contínuo",
            "Encaminhamento p/ serviço espe": "Serviço especializado", "Encaminhamento p/ CAPS": "CAPS"}
DIAGNOSTICOS = {"jogo_cid": "Jogo patológico", "ansiedade_cid": "Ansiedade", "depressao_cid": "Depressão",
                "alcool_cid": "Álcool"}

linhas = []
for grupo, rotulo in DIAGNOSTICOS.items():
    n = int(aps.filter(pl.col("grupo") == grupo)["atendimentos"].sum())
    linha = {"Diagnóstico": rotulo, "Atendimentos": n}
    for nome, coluna in CONDUTAS.items():
        x = int(conduta.filter((pl.col("grupo") == grupo) & (pl.col("conduta") == nome))["n"].sum())
        linha[coluna] = 100 * x / n
    linhas.append(linha)
tabela = pl.DataFrame(linhas)
tabela.with_columns(pl.col(list(CONDUTAS.values())).round(1))

In [ ]:
tabela_tex(
    "tabela1",
    f"Conduta registrada nos atendimentos da atenção primária, jan/2025 a {nome_mes(FIM_APS)} (% dos atendimentos).",
    ["Diagnóstico", "Atendimentos", *[c + " (%)" for c in CONDUTAS.values()]],
    [[r["Diagnóstico"], br(r["Atendimentos"], 0), *[br(r[c], 1) for c in CONDUTAS.values()]] for r in tabela.to_dicts()],
    "Um atendimento pode ter mais de uma conduta, então as linhas não somam 100%. O registro não diz qual é o serviço "
    "especializado (psiquiatria, psicologia ou outro). Fonte: SISAB.",
    "p{2.6cm}R{1.9cm}R{1.4cm}R{1.8cm}R{2.2cm}R{2.2cm}R{1.3cm}")  # R{} = coluna à direita com quebra de linha